# Blockchain Alpha — Random 10k Wallet Longitudinal Study

Tests the core thesis directly: can we find wallets in the general ETH population whose daily buy actions predict positive forward returns?

Uses a random sample from the existing discovery pool — no convexity pre-selection. Fetches 2.5 years of daily balance data per wallet.

**Three-phase pipeline:**
1. Random sample + historical balance probe (~$0.14)
2. Daily balance fetch across full window (~$85)
3. Forward return skill scoring with FDR correction

**Fully restart-safe:** probe checkpoint + daily balance store both support warm-starting after crashes.

**Key question:** Of ~6-10k random wallets with 2.5 years of data, how many show statistically significant forward return predictability? And how does that compare to the expert pool selected by convexity scoring?

## 1. Configuration & imports

In [ ]:
from claude_refactor_core_utils import *
from datetime import date, timedelta
import datetime
import pandas as pd
import numpy as np
import os
import json
import random
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import binom
from statsmodels.stats.multitest import multipletests

# --- Required keys ---
ALCHEMY_API_KEY   = "8J0Ou3_XsuWVbOiZ0xpEvioGx7k0wab9"
ETHERSCAN_API_KEY = "GH6FVBANEIR1HS6PEN36XMYQRY2IDEAAXR"
assert ALCHEMY_API_KEY,   "Set ALCHEMY_API_KEY"
assert ETHERSCAN_API_KEY, "Set ETHERSCAN_API_KEY"

# --- Window ---
HISTORY_START_DATE = date(2024, 1, 1)    # balance history start (slight warmup before train)
FETCH_END_DATE     = date.today() - timedelta(days=21)
TRAIN_END_DATE     = date(2025, 8, 1)    # for period tagging

# --- Sampling ---
TARGET_WALLETS     = 10_000              # target after history probe
OVERSAMPLE_FACTOR  = 8                   # probe this many times target to hit ~10k survivors
RANDOM_SEED        = 42

# --- Signal config ---
MIN_DELTA_ETH      = 0.005
HORIZONS           = [1, 3, 5, 10]      # forward return horizons in days
FDR_ALPHA          = 0.05               # false discovery rate threshold

# --- File paths ---
EXISTING_BLOCK_MAP_CSV    = '/home/colin/projects/blockchain_alpha/data/warm_start_block_df.csv'
DISCOVERY_CHECKPOINT_CSV  = '/home/colin/projects/blockchain_alpha/data/discovery_checkpoint.csv'
PROBE_CHECKPOINT_CSV      = '/home/colin/projects/blockchain_alpha/data/random_10k_probe_checkpoint.csv'
BALANCE_STORE_CSV         = '/home/colin/projects/blockchain_alpha/data/balance_store_random_10k.csv'
QUALIFIED_WALLETS_CSV     = '/home/colin/projects/blockchain_alpha/data/random_10k_qualified_wallets.csv'
SKILL_RESULTS_CSV         = '/home/colin/projects/blockchain_alpha/data/random_10k_skill_results.csv'

print(f"History window: {HISTORY_START_DATE} to {FETCH_END_DATE}")
print(f"Window length:  {(FETCH_END_DATE - HISTORY_START_DATE).days} days")
print(f"Target wallets: {TARGET_WALLETS:,}")
print(f"Oversample:     {TARGET_WALLETS * OVERSAMPLE_FACTOR:,} addresses to probe")


## 2. Build candidate address pool

Union all addresses from the existing discovery checkpoint. Random sample for historical balance probing.

In [ ]:
# ============================================================
# Build candidate pool from existing discovery checkpoint
# Union all hist_active_addr across all discovery days
# ============================================================

if not os.path.exists(DISCOVERY_CHECKPOINT_CSV):
    raise FileNotFoundError(f"Discovery checkpoint not found: {DISCOVERY_CHECKPOINT_CSV}")

checkpoint_df = pd.read_csv(DISCOVERY_CHECKPOINT_CSV)
print(f"Discovery checkpoint: {len(checkpoint_df)} days")

# Union all addresses across all discovery days
all_discovered = set()
for _, row in checkpoint_df.iterrows():
    addrs = json.loads(row['hist_active_addr'])
    all_discovered.update(a.lower().strip() for a in addrs)

print(f"Total unique addresses in discovery pool: {len(all_discovered):,}")
print()

# Random sample for probing
random.seed(RANDOM_SEED)
n_to_probe = min(TARGET_WALLETS * OVERSAMPLE_FACTOR, len(all_discovered))
sample_pool = sorted(all_discovered)
random.shuffle(sample_pool)
addresses_to_probe = sample_pool[:n_to_probe]

print(f"Random sample to probe: {len(addresses_to_probe):,}")
print(f"Expected survivors (~12% pass rate): {int(len(addresses_to_probe) * 0.12):,}")


## 3. Block map

In [ ]:
# Build/update block map
block_df = build_or_update_block_map(
    start_date=str(HISTORY_START_DATE),
    end_date=str(FETCH_END_DATE),
    period="day",
    etherscan_api_key=ETHERSCAN_API_KEY,
    existing_csv_path=EXISTING_BLOCK_MAP_CSV,
    sleep_sec=0.51,
    write_back=True
)
block_map = dict(zip(block_df['timestamp'], block_df['block_number']))
print(f"Block map: {len(block_map):,} dates")


## 4. Phase 1 — Historical balance probe (restart-safe)

Check which addresses have non-zero ETH balance at `HISTORY_START_DATE`. Results saved to `PROBE_CHECKPOINT_CSV` after each batch. **Restart-safe:** already-probed addresses are skipped.

In [ ]:
# ============================================================
# Phase 1: Historical balance probe
# Check which addresses have non-zero ETH balance at HISTORY_START_DATE
# Restart-safe: saves results to probe checkpoint
# ============================================================

def load_probe_checkpoint(path):
    """Load previously probed addresses."""
    if not os.path.exists(path):
        return {}, set()
    df = pd.read_csv(path)
    qualified = set(df[df['has_balance']==True]['address'].tolist())
    probed    = set(df['address'].tolist())
    return qualified, probed

def save_probe_results(path, results_batch):
    """Append batch of probe results to checkpoint."""
    rows = pd.DataFrame(results_batch)
    if os.path.exists(path):
        rows.to_csv(path, mode='a', header=False, index=False)
    else:
        rows.to_csv(path, index=False)

# Load checkpoint
qualified_set, already_probed = load_probe_checkpoint(PROBE_CHECKPOINT_CSV)
print(f"Already probed: {len(already_probed):,} addresses")
print(f"Already qualified: {len(qualified_set):,} addresses")

# Find remaining addresses to probe
remaining = [a for a in addresses_to_probe if a not in already_probed]
print(f"Remaining to probe: {len(remaining):,}")
print()

# Get block number for history start date
block_number_history = block_map.get(HISTORY_START_DATE)
assert block_number_history, f"No block found for {HISTORY_START_DATE}"

if remaining:
    print(f"Probing {len(remaining):,} addresses for balance at {HISTORY_START_DATE}...")
    new_qualified = get_addresses_with_nonzero_balance(
        addresses=remaining,
        block_number=block_number_history,
        alchemy_api_key=ALCHEMY_API_KEY,
        max_addresses=len(remaining)
    )

    # Save results
    batch = [
        {'address': a, 'has_balance': a in set(new_qualified)}
        for a in remaining
    ]
    save_probe_results(PROBE_CHECKPOINT_CSV, batch)
    qualified_set.update(new_qualified)
    print(f"New qualified: {len(new_qualified):,}")
else:
    print("All addresses already probed — loading from checkpoint.")

# Final qualified list
qualified_wallets = sorted(qualified_set)
print()
print(f"Total qualified wallets (non-zero balance at {HISTORY_START_DATE}): {len(qualified_wallets):,}")
print(f"Pass rate: {len(qualified_wallets)/len(addresses_to_probe)*100:.1f}%")

# Sample down to TARGET_WALLETS if we have more
if len(qualified_wallets) > TARGET_WALLETS:
    random.seed(RANDOM_SEED)
    qualified_wallets = sorted(random.sample(qualified_wallets, TARGET_WALLETS))
    print(f"Sampled down to {len(qualified_wallets):,}")

# Save final qualified list
pd.DataFrame({'address': qualified_wallets}).to_csv(QUALIFIED_WALLETS_CSV, index=False)
print(f"Qualified wallets saved: {QUALIFIED_WALLETS_CSV}")


## 5. Phase 2 — Daily balance fetch (restart-safe)

Fetch ETH balance for every qualified wallet on every trading day from `HISTORY_START_DATE` to `FETCH_END_DATE`. Saves to `BALANCE_STORE_CSV` after every date. **Restart-safe:** already-fetched (address, date) pairs are skipped.

In [ ]:
# ============================================================
# Phase 2: Daily balance fetch
# Fetch ETH balance for every qualified wallet on every trading day
# Saves after every date — restart-safe
# ============================================================

# Load price history to get all trading dates
df_price = load_eth_usd_yahoo(start="2020-01-01")
df_price['date'] = pd.to_datetime(df_price['date'])
all_dates = sorted(
    df_price.loc[
        (df_price['date'].dt.date >= HISTORY_START_DATE) &
        (df_price['date'].dt.date <= FETCH_END_DATE),
        'date'
    ].dt.date.tolist()
)
all_dates = [d for d in all_dates if d in block_map]

n_pairs = len(qualified_wallets) * len(all_dates)
cost_est = n_pairs * 19 / 1_000_000 * 0.45

print(f"Qualified wallets:  {len(qualified_wallets):,}")
print(f"Trading dates:      {len(all_dates):,}")
print(f"Total pairs:        {n_pairs:,}")
print(f"Estimated cost:     ${cost_est:.2f}")
print()

def load_already_fetched(store_path):
    """Load set of already-fetched (address, date) pairs."""
    if not os.path.exists(store_path):
        return set()
    df = pd.read_csv(store_path, usecols=['address','date'])
    df['date']    = pd.to_datetime(df['date'], format='mixed').dt.normalize()
    df['address'] = df['address'].str.lower().str.strip()
    return set(zip(df['address'], df['date']))

def filter_schedule(schedule_df, already_fetched):
    """Filter schedule to unfetched pairs."""
    sched = schedule_df.copy()
    sched['date']    = pd.to_datetime(sched['date']).dt.normalize()
    sched['address'] = sched['address'].str.lower().str.strip()
    mask = sched.apply(
        lambda r: (r['address'], r['date']) not in already_fetched, axis=1
    )
    n_skip = (~mask).sum()
    if n_skip > 0:
        print(f"  Skipping {n_skip:,} already-fetched pairs")
    return sched[mask].reset_index(drop=True)

# Load already-fetched pairs
print("Loading already-fetched pairs...")
already_fetched = load_already_fetched(BALANCE_STORE_CSV)
print(f"Already fetched: {len(already_fetched):,} pairs "
      f"({len(already_fetched)/n_pairs*100:.1f}% complete)")
print()

# Fetch day by day
for i, event_date in enumerate(all_dates):
    event_date_str = str(event_date)
    event_ts       = pd.Timestamp(event_date).normalize()

    # Build schedule for this date
    sched_day = pd.DataFrame({
        'address': qualified_wallets,
        'date':    event_ts
    })

    sched_new = filter_schedule(sched_day, already_fetched)

    if not sched_new.empty:
        bal = fetch_balances_for_schedule(
            schedule=sched_new,
            block_df=block_df,
            alchemy_api_key=ALCHEMY_API_KEY,
            batch_size=100,
            max_concurrency=30,
            rps=25.0,
            progress_every=200,
            block_date_col="timestamp",
            label=f"rnd10k-{event_date_str}"
        )

        if not bal.empty:
            # Append to store
            if os.path.exists(BALANCE_STORE_CSV):
                bal.to_csv(BALANCE_STORE_CSV, mode='a', header=False, index=False)
            else:
                bal.to_csv(BALANCE_STORE_CSV, index=False)
            # Update already-fetched cache
            bal['date_norm'] = pd.to_datetime(bal['date'], format='mixed').dt.normalize()
            newly_fetched = set(zip(
                bal['address'].str.lower().str.strip(),
                bal['date_norm']
            ))
            already_fetched.update(newly_fetched)

    # Progress every 30 days
    if (i + 1) % 30 == 0 or (i + 1) == len(all_dates):
        pct = len(already_fetched) / n_pairs * 100
        print(f"[{i+1}/{len(all_dates)}] {event_date_str} — "
              f"{len(already_fetched):,} pairs fetched ({pct:.1f}%)")

print("\nPhase 2 complete.")


## 6. Load & clean balance store

In [ ]:
# ============================================================
# Load and clean the random 10k balance store
# ============================================================

print("Loading balance store...")
raw = pd.read_csv(BALANCE_STORE_CSV)
raw['date']        = pd.to_datetime(raw['date'], format='mixed').dt.normalize()
raw['address']     = raw['address'].str.lower().str.strip()
raw['balance_wei'] = pd.to_numeric(raw['balance_wei'], errors='coerce').fillna(0).astype('int64')
raw['block_number']= pd.to_numeric(raw['block_number'], errors='coerce').astype('Int64')

# Deduplicate
balances = (
    raw
    .sort_values(['address', 'date', 'block_number'])
    .groupby(['address', 'date'], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)
balances['balance_eth'] = balances['balance_wei'] / 1e18

print(f"Balance store: {balances.shape[0]:,} rows")
print(f"Unique addresses: {balances['address'].nunique():,}")
print(f"Date range: {balances['date'].min().date()} to {balances['date'].max().date()}")

expected = balances['address'].nunique() * len(all_dates)
print(f"Coverage: {balances.shape[0]/expected*100:.1f}% of expected pairs")


## 7. Infer daily actions

Buy/sell/hold from day-over-day balance delta. Filter to wallets with >= 20 buy trades.

In [ ]:
# ============================================================
# Infer daily buy/sell/hold actions from balance deltas
# ============================================================

df_pos = balances.sort_values(['address', 'date']).copy()
df_pos['delta_eth'] = df_pos.groupby('address')['balance_eth'].diff()

def classify_action(delta):
    if pd.isna(delta): return None
    if delta >  MIN_DELTA_ETH: return 'buy'
    if delta < -MIN_DELTA_ETH: return 'sell'
    return 'hold'

df_pos['action'] = df_pos['delta_eth'].apply(classify_action)
daily_actions = df_pos.dropna(subset=['action']).copy()

print(f"Daily action observations: {len(daily_actions):,}")
print(f"Unique wallets with actions: {daily_actions['address'].nunique():,}")
print()
print("Action distribution:")
print(daily_actions['action'].value_counts())
print()

# Filter: only keep wallets with enough trades to be statistically meaningful
trade_counts = daily_actions[daily_actions['action']=='buy'].groupby('address').size()
active_wallets = trade_counts[trade_counts >= 20].index.tolist()
print(f"Wallets with >= 20 buy trades: {len(active_wallets):,}")

daily_actions_filtered = daily_actions[daily_actions['address'].isin(active_wallets)].copy()
print(f"Filtered action observations: {len(daily_actions_filtered):,}")


## 8. Merge forward returns

In [ ]:
# ============================================================
# Merge forward returns onto daily actions
# ============================================================

# Load price data and compute forward returns
df_price_full = load_eth_usd_yahoo(start="2020-01-01")
df_price_full['date'] = pd.to_datetime(df_price_full['date'])
df_price_full = df_price_full.sort_values('date').reset_index(drop=True)

for h in HORIZONS:
    df_price_full[f'fwd_return_{h}d']    = (
        df_price_full['close'].shift(-h) - df_price_full['close']
    ) / df_price_full['close'] * 100
    df_price_full[f'fwd_direction_{h}d'] = np.sign(df_price_full[f'fwd_return_{h}d'])

df_price_full['period'] = df_price_full['date'].apply(
    lambda d: 'train' if d.date() < TRAIN_END_DATE
              else ('test' if d.date() <= FETCH_END_DATE else 'future')
)

# Merge
trade_df = daily_actions_filtered.merge(
    df_price_full[['date','close','period'] +
                   [f'fwd_return_{h}d' for h in HORIZONS] +
                   [f'fwd_direction_{h}d' for h in HORIZONS]],
    on='date', how='inner'
)

print(f"Trade observations with price data: {len(trade_df):,}")
print(f"Unique wallets: {trade_df['address'].nunique():,}")
print(f"Train obs: {(trade_df['period']=='train').sum():,}")
print(f"Test obs:  {(trade_df['period']=='test').sum():,}")


## 9. Wallet-level skill scoring

For each wallet at each horizon: t-test of buy forward returns vs population baseline. FDR correction (Benjamini-Hochberg) applied per horizon. With 900+ days of data per wallet, individual-level tests now have real statistical power.

In [ ]:
# ============================================================
# Wallet-level forward return skill scoring
# For each wallet: do their buys precede positive returns?
# Test against population baseline, apply FDR correction
# ============================================================

print("Computing wallet-level skill scores...")
print()

skill_rows = []

# Population baseline forward returns
for h in HORIZONS:
    col = f'fwd_return_{h}d'
    baseline_mean = trade_df[col].mean()
    baseline_dir  = (trade_df[f'fwd_direction_{h}d'] > 0).mean()

    # Per-wallet analysis
    for addr, grp in trade_df.groupby('address'):
        buys  = grp[grp['action'] == 'buy']
        sells = grp[grp['action'] == 'sell']

        if len(buys) < 10:
            continue

        buy_returns = buys[col].dropna()
        base_returns = trade_df[col].dropna()

        # t-test: buy forward returns vs population
        t_stat, p_val = stats.ttest_ind(buy_returns, base_returns)

        # Direction accuracy
        buy_dir_acc = (buys[f'fwd_direction_{h}d'].dropna() > 0).mean()

        skill_rows.append({
            'address':      addr,
            'horizon':      h,
            'n_buy_trades': len(buys),
            'n_sell_trades':len(sells),
            'buy_mean_return': buy_returns.mean(),
            'baseline_mean':   baseline_mean,
            'return_diff':  buy_returns.mean() - baseline_mean,
            'dir_accuracy': buy_dir_acc,
            'baseline_dir': baseline_dir,
            'p_value_raw':  p_val,
        })

skill_df = pd.DataFrame(skill_rows)

# Apply FDR correction per horizon
print("Applying FDR correction per horizon...")
for h in HORIZONS:
    mask = skill_df['horizon'] == h
    pvals = skill_df.loc[mask, 'p_value_raw'].values
    reject, pvals_fdr, _, _ = multipletests(pvals, alpha=FDR_ALPHA, method='fdr_bh')
    skill_df.loc[mask, 'p_value_fdr'] = pvals_fdr
    skill_df.loc[mask, 'significant_fdr'] = reject
    skill_df.loc[mask, 'significant_raw'] = pvals < 0.05

    n_sig_fdr = reject.sum()
    n_sig_raw = (pvals < 0.05).sum()
    n_wallets = mask.sum()
    n_expected = int(n_wallets * 0.05)

    print(f"  {h}d horizon ({n_wallets:,} wallets tested):")
    print(f"    Significant (FDR corrected): {n_sig_fdr}")
    print(f"    Significant (uncorrected p<0.05): {n_sig_raw} (random expected: ~{n_expected})")

print()

# Positive-skill wallets: significant FDR AND positive return diff
positive_skill = skill_df[
    skill_df['significant_fdr'] &
    (skill_df['return_diff'] > 0)
]
print(f"Wallets with positive, FDR-significant buy skill:")
print(f"  Across any horizon: {positive_skill['address'].nunique():,}")
for h in HORIZONS:
    n = positive_skill[positive_skill['horizon']==h]['address'].nunique()
    print(f"  At {h}d horizon: {n}")


## 10. Skill distribution visualization

Two panels per horizon: return diff distribution (skilled wallets highlighted) and p-value distribution (uniform = no signal, left-skewed = signal present).

In [ ]:
# ============================================================
# Visualize skill distribution across the random wallet pool
# ============================================================

fig, axes = plt.subplots(2, len(HORIZONS), figsize=(20, 10))

for col_idx, h in enumerate(HORIZONS):
    df_h = skill_df[skill_df['horizon']==h].copy()

    # Row 1: Return diff distribution
    ax = axes[0, col_idx]
    sig_fdr  = df_h[df_h['significant_fdr'] & (df_h['return_diff']>0)]['return_diff']
    not_sig  = df_h[~df_h['significant_fdr']]['return_diff']

    ax.hist(not_sig, bins=50, alpha=0.6, color='gray',
            label=f'Not sig (n={len(not_sig):,})', density=True)
    if len(sig_fdr) > 0:
        ax.hist(sig_fdr, bins=20, alpha=0.8, color='steelblue',
                label=f'Sig+positive (n={len(sig_fdr):,})', density=True)
    ax.axvline(0, color='red', linewidth=1.5, linestyle='--')
    ax.set_title(f'{h}d return diff distribution')
    ax.set_xlabel('Return diff vs baseline (%)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # Row 2: p-value distribution (should be uniform if no signal)
    ax = axes[1, col_idx]
    ax.hist(df_h['p_value_raw'], bins=50, edgecolor='black', color='coral', alpha=0.7)
    ax.axhline(len(df_h)/50, color='red', linewidth=1.5, linestyle='--',
               label='Expected uniform')
    ax.set_title(f'{h}d p-value distribution')
    ax.set_xlabel('p-value (raw)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Random 10k Wallet Pool — Skill Distribution by Horizon', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 11. Top skilled wallets

Detail view of wallets that survive FDR correction with positive return differential.

In [ ]:
# ============================================================
# Examine top skilled wallets in detail
# ============================================================

# Best horizon: pick the one with most FDR-significant positive wallets
best_h = (positive_skill.groupby('horizon')['address']
          .nunique().idxmax() if len(positive_skill) > 0 else HORIZONS[0])

print(f"Best horizon (most significant positive wallets): {best_h}d")
print()

top_wallets = (
    skill_df[
        (skill_df['horizon']==best_h) &
        skill_df['significant_fdr'] &
        (skill_df['return_diff']>0)
    ]
    .sort_values('return_diff', ascending=False)
    .head(30)
)

if len(top_wallets) > 0:
    print(f"Top skilled wallets at {best_h}d horizon:")
    print(top_wallets[['address','n_buy_trades','buy_mean_return',
                        'baseline_mean','return_diff','dir_accuracy',
                        'p_value_raw','p_value_fdr']].to_string())
else:
    print("No FDR-significant positive wallets found.")
    print()
    print("Top wallets by raw return diff (uncorrected):")
    top_raw = (
        skill_df[
            (skill_df['horizon']==best_h) &
            skill_df['significant_raw'] &
            (skill_df['return_diff']>0)
        ]
        .sort_values('return_diff', ascending=False)
        .head(20)
    )
    print(top_raw[['address','n_buy_trades','buy_mean_return',
                   'baseline_mean','return_diff','dir_accuracy','p_value_raw']].to_string())


## 12. Pool-level comparison

Does the random pool collectively beat baseline? Compare to the expert pool results to assess whether convexity selection adds value beyond random sampling.

In [ ]:
# ============================================================
# Pool-level significance test on random wallets
# Do they collectively beat baseline? (comparison to expert pool)
# ============================================================

print("=== POOL-LEVEL TEST: RANDOM WALLETS vs EXPERT POOL ===")
print()

buys_all = trade_df[trade_df['action']=='buy']

for h in HORIZONS:
    col = f'fwd_return_{h}d'
    base_ret = trade_df[col].dropna()
    buy_ret  = buys_all[col].dropna()

    t, p = stats.ttest_ind(buy_ret, base_ret)
    diff = buy_ret.mean() - base_ret.mean()
    dir_acc = (buys_all[f'fwd_direction_{h}d'].dropna() > 0).mean()

    print(f"  {h}d: n_buys={len(buy_ret):,}, diff={diff:+.3f}%, "
          f"dir_acc={dir_acc:.3f}, p={p:.4f}")

print()
print("Compare to expert pool (from pool-level tests):")
print("  Buy pool (train): diff ~+0.07%, multiple 1.36x, p≈0")
print("  Buy pool (test):  diff ~+0.07%, multiple 1.38x, p≈0")
print()
print("If random pool shows similar or stronger signal, convexity selection")
print("adds no value. If random pool is weaker/random, selection is validated.")


## 13. Save results

In [ ]:
skill_df.to_csv(SKILL_RESULTS_CSV, index=False)
print(f"Skill results saved: {SKILL_RESULTS_CSV}")
print(f"Shape: {skill_df.shape}")
print()

# Final summary
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"Qualified wallets (history probe):  {len(qualified_wallets):,}")
print(f"Active wallets (>= 20 buy trades):  {len(active_wallets):,}")
print()
print("FDR-significant positive-skill wallets:")
for h in HORIZONS:
    n = len(positive_skill[positive_skill['horizon']==h])
    pct = n / skill_df[skill_df['horizon']==h]['address'].nunique() * 100
    print(f"  {h}d horizon: {n} wallets ({pct:.2f}%)")
print()

# Key comparison
n_active = skill_df['address'].nunique()
n_sig_any = positive_skill['address'].nunique()
print(f"Wallets significant on any horizon: {n_sig_any} / {n_active} "
      f"({n_sig_any/n_active*100:.2f}% of active wallets)")
print()
if n_sig_any > 0:
    print("RESULT: Skilled wallets exist in random population.")
    print("Next step: compare skill profile to expert pool.")
else:
    print("RESULT: No individually significant wallets in random pool.")
    print("This validates convexity selection OR suggests daily skill")
    print("is not detectable at individual level regardless of selection.")


---
## Extension: Top 500 Wallets — Extended History to 2021

Takes the top 500 wallets by raw 1d return differential from the random 10k study and fetches their full history back to January 2021 (~4+ years). More observations per wallet = more statistical power = cleaner signal separation.

**Estimated cost: ~$3.75. Restart-safe.**

### E1. Extended configuration

In [ ]:
# ============================================================
# EXTENSION: Top 500 wallets — extended history to 2021
# Appended to random_10k_study.ipynb
# ============================================================

EXTENDED_START_DATE    = date(2021, 1, 1)
EXTENDED_END_DATE      = FETCH_END_DATE   # reuse from main notebook
EXTENDED_BALANCE_CSV   = '/home/colin/projects/blockchain_alpha/data/balance_store_top500_extended.csv'
EXTENDED_SKILL_CSV     = '/home/colin/projects/blockchain_alpha/data/top500_extended_skill_results.csv'
N_TOP_WALLETS          = 500

print(f"Extended window: {EXTENDED_START_DATE} to {EXTENDED_END_DATE}")
print(f"Window length:   {(EXTENDED_END_DATE - EXTENDED_START_DATE).days} days (~{(EXTENDED_END_DATE - EXTENDED_START_DATE).days/365:.1f} years)")

# Cost estimate
n_days = len([d for d in all_dates if d >= EXTENDED_START_DATE])
cost_est = N_TOP_WALLETS * n_days * 19 / 1_000_000 * 0.45
print(f"Estimated cost:  ${cost_est:.2f}")


### E2. Select top 500 wallets

In [ ]:
# Select top 500 wallets by raw return diff at 1d horizon
# Using skill_df from the main notebook (already computed)

top500_df = (
    skill_df[skill_df['horizon'] == 1]
    .sort_values('return_diff', ascending=False)
    .head(N_TOP_WALLETS)
    .copy()
)

top500_wallets = top500_df['address'].tolist()

print(f"Top {N_TOP_WALLETS} wallets selected by 1d return diff")
print(f"Return diff range: {top500_df['return_diff'].min():.3f}% to {top500_df['return_diff'].max():.3f}%")
print(f"Dir accuracy range: {top500_df['dir_accuracy'].min():.3f} to {top500_df['dir_accuracy'].max():.3f}")
print(f"p-value range: {top500_df['p_value_raw'].min():.4f} to {top500_df['p_value_raw'].max():.4f}")
print()
print("Distribution of return diffs:")
print(top500_df['return_diff'].describe().round(3))


### E3. Existence probe — did they exist in 2021?

In [ ]:
# Probe: which of the top 500 existed in 2021?
# Only fetch extended history for wallets with non-zero balance at EXTENDED_START_DATE

EXTENDED_PROBE_CSV = '/home/colin/projects/blockchain_alpha/data/top500_existence_probe.csv'

def load_existence_probe(path):
    if not os.path.exists(path):
        return {}, set()
    df = pd.read_csv(path)
    qualified = set(df[df['has_balance']==True]['address'].tolist())
    probed    = set(df['address'].tolist())
    return qualified, probed

def save_existence_probe(path, results):
    rows = pd.DataFrame(results)
    if os.path.exists(path):
        rows.to_csv(path, mode='a', header=False, index=False)
    else:
        rows.to_csv(path, index=False)

existing_qualified, already_probed = load_existence_probe(EXTENDED_PROBE_CSV)
remaining = [a for a in top500_wallets if a not in already_probed]

print(f"Already probed: {len(already_probed)}")
print(f"Already qualified: {len(existing_qualified)}")
print(f"Remaining to probe: {len(remaining)}")

if remaining:
    # Get block number for extended start date
    block_2021 = block_map.get(EXTENDED_START_DATE)
    if not block_2021:
        # Build extended block map if needed
        print("Extending block map to cover 2021...")
        block_df_ext = build_or_update_block_map(
            start_date=str(EXTENDED_START_DATE),
            end_date=str(EXTENDED_END_DATE),
            period="day",
            etherscan_api_key=ETHERSCAN_API_KEY,
            existing_csv_path=EXISTING_BLOCK_MAP_CSV,
            sleep_sec=0.51,
            write_back=True
        )
        block_map.update(dict(zip(block_df_ext['timestamp'], block_df_ext['block_number'])))
        block_2021 = block_map.get(EXTENDED_START_DATE)

    assert block_2021, f"No block found for {EXTENDED_START_DATE}"
    print(f"Block at {EXTENDED_START_DATE}: {block_2021:,}")

    new_qualified = get_addresses_with_nonzero_balance(
        addresses=remaining,
        block_number=block_2021,
        alchemy_api_key=ALCHEMY_API_KEY,
        max_addresses=len(remaining)
    )

    batch = [
        {'address': a, 'has_balance': a in set(new_qualified)}
        for a in remaining
    ]
    save_existence_probe(EXTENDED_PROBE_CSV, batch)
    existing_qualified.update(set(new_qualified))
    print(f"New qualified (existed in 2021): {len(new_qualified)}")

extended_wallets = sorted(existing_qualified)
print()
print(f"Wallets with history back to {EXTENDED_START_DATE}: {len(extended_wallets)} of {N_TOP_WALLETS}")
print(f"Wallets that didn't exist in 2021: {N_TOP_WALLETS - len(extended_wallets)}")
print(f"Will fetch from their earliest available date instead")


### E4. Extended history fetch (restart-safe)

Fetches pre-2024 dates only — post-2024 data already exists in the random_10k store. Saves to `balance_store_top500_extended.csv`.

In [ ]:
# ============================================================
# Extended history fetch — top 500 wallets, 2021 to present
# Merges with existing random_10k balance data where available
# Saves to separate EXTENDED_BALANCE_CSV
# Restart-safe: skips already-fetched pairs
# ============================================================

# Build full date list from extended start
df_price_full2 = load_eth_usd_yahoo(start="2020-01-01")
df_price_full2['date'] = pd.to_datetime(df_price_full2['date'])

extended_dates = sorted(
    df_price_full2.loc[
        (df_price_full2['date'].dt.date >= EXTENDED_START_DATE) &
        (df_price_full2['date'].dt.date <= EXTENDED_END_DATE),
        'date'
    ].dt.date.tolist()
)
extended_dates = [d for d in extended_dates if d in block_map]

# For wallets that didn't exist in 2021, fetch from HISTORY_START_DATE
wallets_from_2021   = set(extended_wallets)
wallets_from_2024   = set(top500_wallets) - wallets_from_2021

print(f"Fetch window 2021+: {len(wallets_from_2021)} wallets × {len(extended_dates)} dates")
print(f"Fetch window 2024+: {len(wallets_from_2024)} wallets × {len(all_dates)} dates")
print()

# Load already-fetched pairs from EXISTING random_10k store
# These cover HISTORY_START_DATE to FETCH_END_DATE for all active wallets
# We can reuse them rather than re-fetching
def load_fetched_pairs(path):
    if not os.path.exists(path):
        return set()
    df = pd.read_csv(path, usecols=['address','date'])
    df['date']    = pd.to_datetime(df['date'], format='mixed').dt.normalize()
    df['address'] = df['address'].str.lower().str.strip()
    return set(zip(df['address'], df['date']))

print("Loading already-fetched pairs from random_10k store...")
already_fetched_10k = load_fetched_pairs(BALANCE_STORE_CSV)
print(f"  random_10k store: {len(already_fetched_10k):,} pairs")

# Also load extended store if it exists
already_fetched_ext = load_fetched_pairs(EXTENDED_BALANCE_CSV)
print(f"  extended store:   {len(already_fetched_ext):,} pairs")

all_already_fetched = already_fetched_10k | already_fetched_ext
print(f"  combined:         {len(all_already_fetched):,} pairs")
print()

def filter_unfetched(addresses, dates, already_fetched):
    """Return list of (address, date) pairs not yet fetched."""
    pairs = []
    for d in dates:
        d_ts = pd.Timestamp(d).normalize()
        for a in addresses:
            if (a.lower().strip(), d_ts) not in already_fetched:
                pairs.append((a, d_ts))
    return pairs

# Build schedule for 2021+ wallets (new dates only)
new_dates_2021 = [d for d in extended_dates if d < HISTORY_START_DATE]
print(f"New dates to fetch (pre-2024): {len(new_dates_2021)}")

total_new = len(wallets_from_2021) * len(new_dates_2021) + len(wallets_from_2024) * len(all_dates)
cost_new  = total_new * 19 / 1_000_000 * 0.45
print(f"New pairs to fetch: ~{total_new:,} (est. ${cost_new:.2f})")
print()

# Fetch pre-2024 dates for 2021+ wallets
fetch_count = 0
for i, event_date in enumerate(new_dates_2021):
    event_ts       = pd.Timestamp(event_date).normalize()
    event_date_str = str(event_date)

    sched_day = pd.DataFrame({
        'address': list(wallets_from_2021),
        'date':    event_ts
    })
    sched_day['date']    = pd.to_datetime(sched_day['date']).dt.normalize()
    sched_day['address'] = sched_day['address'].str.lower().str.strip()
    mask = sched_day.apply(
        lambda r: (r['address'], r['date']) not in all_already_fetched, axis=1
    )
    sched_new = sched_day[mask].reset_index(drop=True)

    if not sched_new.empty:
        bal = fetch_balances_for_schedule(
            schedule=sched_new,
            block_df=block_df if hasattr(block_df, 'columns') else block_df_ext,
            alchemy_api_key=ALCHEMY_API_KEY,
            batch_size=100,
            max_concurrency=30,
            rps=25.0,
            progress_every=500,
            block_date_col="timestamp",
            label=f"ext-{event_date_str}"
        )

        if not bal.empty:
            if os.path.exists(EXTENDED_BALANCE_CSV):
                bal.to_csv(EXTENDED_BALANCE_CSV, mode='a', header=False, index=False)
            else:
                bal.to_csv(EXTENDED_BALANCE_CSV, index=False)

            bal['date_norm'] = pd.to_datetime(bal['date'], format='mixed').dt.normalize()
            newly = set(zip(
                bal['address'].str.lower().str.strip(),
                bal['date_norm']
            ))
            all_already_fetched.update(newly)
            fetch_count += len(bal)

    if (i + 1) % 30 == 0 or (i + 1) == len(new_dates_2021):
        print(f"[{i+1}/{len(new_dates_2021)}] {event_date_str} — {fetch_count:,} new rows fetched")

print(f"\nPre-2024 fetch complete. {fetch_count:,} new rows.")
print("Post-2024 data already exists in random_10k store — will merge in analysis.")


### E5. Merge extended + existing data, infer actions

In [ ]:
# ============================================================
# Merge extended + existing data, recompute skill scores
# ============================================================

print("Loading and merging balance data...")

# Load extended store (pre-2024 data)
if os.path.exists(EXTENDED_BALANCE_CSV):
    ext_raw = pd.read_csv(EXTENDED_BALANCE_CSV)
    ext_raw['date']        = pd.to_datetime(ext_raw['date'], format='mixed').dt.normalize()
    ext_raw['address']     = ext_raw['address'].str.lower().str.strip()
    ext_raw['balance_wei'] = pd.to_numeric(ext_raw['balance_wei'], errors='coerce').fillna(0).astype('int64')
    print(f"Extended store (pre-2024): {len(ext_raw):,} rows")
else:
    ext_raw = pd.DataFrame()
    print("No extended store found — using existing data only")

# Load existing random_10k store (post-2024 data) for top 500 wallets only
existing_raw = pd.read_csv(BALANCE_STORE_CSV)
existing_raw['date']    = pd.to_datetime(existing_raw['date'], format='mixed').dt.normalize()
existing_raw['address'] = existing_raw['address'].str.lower().str.strip()
existing_raw['balance_wei'] = pd.to_numeric(existing_raw['balance_wei'], errors='coerce').fillna(0).astype('int64')

# Filter to top 500 only
top500_set = set(a.lower().strip() for a in top500_wallets)
existing_filtered = existing_raw[existing_raw['address'].isin(top500_set)].copy()
print(f"Existing store (post-2024, top500 only): {len(existing_filtered):,} rows")

# Combine
if not ext_raw.empty:
    combined_raw = pd.concat([ext_raw, existing_filtered], ignore_index=True)
else:
    combined_raw = existing_filtered.copy()

# Deduplicate
combined = (
    combined_raw
    .sort_values(['address', 'date', 'block_number'])
    .groupby(['address', 'date'], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)
combined['balance_eth'] = pd.to_numeric(combined['balance_wei'], errors='coerce') / 1e18

print(f"Combined deduplicated: {len(combined):,} rows")
print(f"Unique addresses: {combined['address'].nunique():,}")
print(f"Date range: {combined['date'].min().date()} to {combined['date'].max().date()}")

# Infer actions
combined = combined.sort_values(['address', 'date'])
combined['delta_eth'] = combined.groupby('address')['balance_eth'].diff()
combined['action'] = combined['delta_eth'].apply(classify_action)
actions_ext = combined.dropna(subset=['action']).copy()

print(f"\nAction observations: {len(actions_ext):,}")
print(actions_ext['action'].value_counts())

# Per-wallet trade count summary
trade_counts_ext = actions_ext[actions_ext['action']=='buy'].groupby('address').size()
print(f"\nBuy trade count distribution (extended window):")
print(trade_counts_ext.describe().round(1))
print(f"Wallets with >= 20 buy trades: {(trade_counts_ext >= 20).sum()}")
print(f"Wallets with >= 50 buy trades: {(trade_counts_ext >= 50).sum()}")
print(f"Wallets with >= 100 buy trades: {(trade_counts_ext >= 100).sum()}")


### E6. Recompute skill scores on extended window

In [ ]:
# ============================================================
# Recompute forward return skill on extended dataset
# ============================================================

# Merge forward returns
df_price_ext = df_price_full2.copy()
for h in HORIZONS:
    df_price_ext[f'fwd_return_{h}d']    = (
        df_price_ext['close'].shift(-h) - df_price_ext['close']
    ) / df_price_ext['close'] * 100
    df_price_ext[f'fwd_direction_{h}d'] = np.sign(df_price_ext[f'fwd_return_{h}d'])

df_price_ext['period'] = df_price_ext['date'].apply(
    lambda d: 'pre_train' if d.date() < TRAIN_START_DATE
              else ('train' if d.date() < TRAIN_END_DATE
              else ('test' if d.date() <= EXTENDED_END_DATE else 'future'))
)

trade_ext = actions_ext.merge(
    df_price_ext[['date','period'] +
                  [f'fwd_return_{h}d' for h in HORIZONS] +
                  [f'fwd_direction_{h}d' for h in HORIZONS]],
    on='date', how='inner'
)

# Filter to >= 20 buy trades
active_ext = trade_counts_ext[trade_counts_ext >= 20].index.tolist()
trade_ext  = trade_ext[trade_ext['address'].isin(active_ext)].copy()

print(f"Trade observations (extended, active wallets): {len(trade_ext):,}")
print(f"Active wallets: {trade_ext['address'].nunique():,}")
print()

# Vectorized skill scoring
print("Computing extended skill scores...")
skill_rows_ext = []

buys_ext  = trade_ext[trade_ext['action']=='buy'].copy()
sells_ext = trade_ext[trade_ext['action']=='sell'].copy()
buy_grps  = buys_ext.groupby('address')

for h in HORIZONS:
    col     = f'fwd_return_{h}d'
    dir_col = f'fwd_direction_{h}d'

    base_mean = trade_ext[col].mean()
    base_std  = trade_ext[col].std()
    base_n    = trade_ext[col].notna().sum()
    base_dir  = (trade_ext[dir_col].dropna() > 0).mean()

    print(f"  {h}d horizon...")

    for addr in active_ext:
        if addr not in buy_grps.groups:
            continue
        buy_ret = buy_grps.get_group(addr)[col].dropna()
        if len(buy_ret) < 10:
            continue

        buy_mean = buy_ret.mean()
        buy_std  = buy_ret.std()
        buy_n    = len(buy_ret)

        se = np.sqrt(buy_std**2/buy_n + base_std**2/base_n)
        if se == 0:
            p_val = 1.0
        else:
            t_stat = (buy_mean - base_mean) / se
            df_w   = (buy_std**2/buy_n + base_std**2/base_n)**2 /                      ((buy_std**2/buy_n)**2/(buy_n-1) + (base_std**2/base_n)**2/(base_n-1))
            p_val  = 2 * stats.t.sf(abs(t_stat), df=df_w)

        buy_dir_acc = (buy_grps.get_group(addr)[dir_col].dropna() > 0).mean()

        n_sell = len(sells_ext[sells_ext['address']==addr]) if addr in sells_ext['address'].values else 0

        skill_rows_ext.append({
            'address':        addr,
            'horizon':        h,
            'n_buy_trades':   buy_n,
            'n_sell_trades':  n_sell,
            'buy_mean_return':buy_mean,
            'baseline_mean':  base_mean,
            'return_diff':    buy_mean - base_mean,
            'dir_accuracy':   buy_dir_acc,
            'baseline_dir':   base_dir,
            'p_value_raw':    p_val,
        })

skill_ext_df = pd.DataFrame(skill_rows_ext)

# FDR correction
print("\nApplying FDR correction...")
for h in HORIZONS:
    mask   = skill_ext_df['horizon'] == h
    pvals  = skill_ext_df.loc[mask, 'p_value_raw'].values
    reject, pvals_fdr, _, _ = multipletests(pvals, alpha=FDR_ALPHA, method='fdr_bh')
    skill_ext_df.loc[mask, 'p_value_fdr']     = pvals_fdr
    skill_ext_df.loc[mask, 'significant_fdr'] = reject.astype(bool)
    skill_ext_df.loc[mask, 'significant_raw'] = (pvals < 0.05)

    n_sig = reject.sum()
    n_raw = (pvals < 0.05).sum()
    n_tot = mask.sum()
    print(f"  {h}d: {n_sig} FDR-significant, {n_raw} uncorrected p<0.05 (of {n_tot} wallets)")

positive_ext = skill_ext_df[
    skill_ext_df['significant_fdr'] & (skill_ext_df['return_diff'] > 0)
]
print(f"\nFDR-significant positive wallets (any horizon): {positive_ext['address'].nunique()}")


### E7. Results — tiering and comparison to 2.5yr window

Key question: do the top wallets' signals strengthen or weaken with more data? Strengthening = genuine skill. Weakening = lucky over 2.5 years.

In [ ]:
# ============================================================
# Extended results: top wallets and tiering
# ============================================================

best_h_ext = 1  # use 1d as primary horizon

# Full ranking at 1d horizon
ranking_1d = (
    skill_ext_df[skill_ext_df['horizon']==best_h_ext]
    .sort_values('return_diff', ascending=False)
    .copy()
)

print(f"=== TOP 30 WALLETS — EXTENDED WINDOW ({best_h_ext}d horizon) ===")
print(ranking_1d[['address','n_buy_trades','buy_mean_return','baseline_mean',
                   'return_diff','dir_accuracy','p_value_raw','p_value_fdr',
                   'significant_fdr']].head(30).to_string())
print()

# Tiering
tier1 = ranking_1d[ranking_1d['significant_fdr'] & (ranking_1d['return_diff']>0)]
tier2 = ranking_1d[
    ~ranking_1d['significant_fdr'] &
    (ranking_1d['p_value_raw'] < 0.05) &
    (ranking_1d['return_diff'] > 0)
]
tier3 = ranking_1d[
    ~ranking_1d['significant_fdr'] &
    (ranking_1d['p_value_raw'] >= 0.05)
]

print(f"Tier 1 — FDR-significant positive skill: {len(tier1)} wallets")
print(f"Tier 2 — Uncorrected significant, positive: {len(tier2)} wallets")
print(f"Tier 3 — Not significant: {len(tier3)} wallets")
print()

# Compare to original 2.5-year results
orig_top = skill_df[skill_df['horizon']==best_h_ext].sort_values('return_diff', ascending=False).head(500)
compare = orig_top[['address','return_diff','p_value_raw']].merge(
    ranking_1d[['address','return_diff','n_buy_trades','p_value_raw','significant_fdr']],
    on='address', how='left', suffixes=('_2.5yr','_extended')
)

print("=== COMPARISON: 2.5yr vs extended window for top 30 ===")
print(compare[['address','return_diff_2.5yr','return_diff_extended',
               'n_buy_trades','p_value_raw_2.5yr','p_value_raw_extended',
               'significant_fdr']].head(30).to_string())
print()
print("Wallets whose signal STRENGTHENED with more data:")
strengthened = compare[
    (compare['return_diff_extended'] > compare['return_diff_2.5yr']) &
    (compare['return_diff_extended'] > 0)
]
print(f"  {len(strengthened)} wallets")
print()
print("Wallets whose signal WEAKENED with more data (likely lucky over 2.5yr):")
weakened = compare[
    (compare['return_diff_extended'] < compare['return_diff_2.5yr'] * 0.5) |
    (compare['return_diff_extended'] < 0)
]
print(f"  {len(weakened)} wallets")


### E8. Save extended results

In [ ]:
skill_ext_df.to_csv(EXTENDED_SKILL_CSV, index=False)
print(f"Extended skill results saved: {EXTENDED_SKILL_CSV}")
print()

# Final summary
print("=" * 60)
print("EXTENDED STUDY — FINAL SUMMARY")
print("=" * 60)
print(f"Window: {EXTENDED_START_DATE} to {EXTENDED_END_DATE}")
print(f"Wallets analyzed: {skill_ext_df['address'].nunique():,}")
print()
print("FDR-significant positive-skill wallets:")
for h in HORIZONS:
    n = len(positive_ext[positive_ext['horizon']==h])
    tot = skill_ext_df[skill_ext_df['horizon']==h]['address'].nunique()
    print(f"  {h}d: {n} / {tot} ({n/tot*100:.2f}%)")
print()

if len(tier1) > 0:
    print("RESULT: Confirmed skilled wallets found.")
    print("Tier 1 wallet addresses (model-ready):")
    for addr in tier1['address'].tolist():
        row = tier1[tier1['address']==addr].iloc[0]
        print(f"  {addr}  diff={row['return_diff']:+.3f}%  "
              f"dir={row['dir_accuracy']:.3f}  n={row['n_buy_trades']}  "
              f"p_fdr={row['p_value_fdr']:.5f}")
else:
    print("No FDR-significant wallets found in extended window.")
    print("Top 10 by return diff (Tier 2 candidates):")
    print(ranking_1d[['address','n_buy_trades','return_diff',
                       'dir_accuracy','p_value_raw']].head(10).to_string())
